# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hariommishra-12/Flyrank-Project/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip install -q duckdb huggingface_hub

import duckdb, os
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}');")

BASE = "hf://datasets/FlyRank/internship-warehouse"
tbl = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')"
print("connected")

connected


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below
One row = one (report_date, client_hash_id, content_hash_id) in
`fact_content_daily_performance` — a single content item's performance on a single day, for a
single client. I'm iterating on the mid-panel partition `month=2026-03` (per the repo's
warning: the `_sample` table is the final month, June 2026 — the natural outcome window of any
past→future label — so it's sealed for testing only, never for developing label logic)..*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Schema first -- I don't assume column names, I check them.
con.sql(f"DESCRIBE SELECT * FROM {tbl} LIMIT 1").show()

con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {tbl}
""").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────┬────────────┐
│ n_rows  │  min_date  │  max_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why
**Feature** (known before the decision point): gsc_avg_position, [gsc impressions/clicks-style
columns from DESCRIBE], sessions_ai, ga4_* engagement columns (only where
ga4_data_available is TRUE).

**Label/proxy**: none shipped — I'll build one myself from prior-window vs. future-window
comparison; never a pre-computed trend/direction-style bucket, since that's the same label
trap as the starter CSV.

**Context**: report_date, client_hash_id, content_hash_id — joining/grouping only, never a
feature.

**Excluded**: ga4_* columns where ga4_data_available = FALSE (zero-filled, not real zero
engagement) — [state why for any other column you exclude, once you see the schema]..*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Pull real rows so the classification above is checked against actual values.
con.sql(f"SELECT * FROM {tbl} LIMIT 3").show()

# How much of this month is excluded (ga4 not yet tracking)? -- the number
# that justifies the "Excluded" bucket above.
con.sql(f"""
    SELECT
        ga4_data_available,
        COUNT(*) AS n_rows,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_month
    FROM {tbl}
    GROUP BY 1
""").show()

# Confirm the two ID columns behave like join keys, not features.
con.sql(f"""
    SELECT
        COUNT(DISTINCT client_hash_id) AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM {tbl}
""").show()

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬──────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_claude

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬─────────┬───────────┬──────────────┐
│ ga4_data_available │ n_rows  │ n_clients │ pct_of_month │
│      boolean       │  int64  │   int64   │    double    │
├────────────────────┼─────────┼───────────┼──────────────┤
│ NULL               │ 3018741 │        22 │         30.7 │
│ false              │ 6408671 │        43 │         65.1 │
│ true               │  413966 │        41 │          4.2 │
└────────────────────┴─────────┴───────────┴──────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬─────────────────┐
│ n_clients │ n_content_items │
│   int64   │      int64      │
├───────────┼─────────────────┤
│        55 │          331437 │
└───────────┴─────────────────┘



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess
Three checks: the grain claim from Section 1 (no duplicate report_date × client × content
rows), missingness in the main position signal (and whether it's patterned by
ga4_data_available, not random), and per-client date ranges (since histories don't start
together — an unbalanced panel)..*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1) Grain check -- should return 0 rows if the grain holds.
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM {tbl} GROUP BY 1,2,3 HAVING c > 1 LIMIT 5
""").show()

# 2) Missingness, overall and by ga4_data_available (checks for a *pattern*, not just a rate).
con.sql(f"""
    SELECT ga4_data_available, COUNT(*) n,
           AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS pct_missing_position
    FROM {tbl} GROUP BY 1
""").show()

# 3) Per-client window check -- histories don't start together.
con.sql(f"""
    SELECT client_hash_id, MIN(report_date) min_d, MAX(report_date) max_d, COUNT(*) n
    FROM {tbl} GROUP BY 1 ORDER BY n DESC LIMIT 10
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬─────────┬──────────────────────┐
│ ga4_data_available │    n    │ pct_missing_position │
│      boolean       │  int64  │        double        │
├────────────────────┼─────────┼──────────────────────┤
│ NULL               │ 3018741 │  0.49370747606369675 │
│ false              │ 6408671 │    0.731871397361481 │
│ true               │  413966 │  0.11986250078508863 │
└────────────────────┴─────────┴──────────────────────┘

┌─────────────────────────┬────────────┬────────────┬────────┐
│     client_hash_id      │   min_d    │   max_d    │   n    │
│         varchar         │    date    │    date    │ int64  │
├─────────────────────────┼────────────┼────────────┼────────┤
│ client_625b6439094e23e4 │ 2026-03-01 │ 2026-03-31 │ 988497 │
│ client_3ffa76342f366962 │ 2026-03-01 │ 2026-03-31 │ 904847 │
│ client_73cda7b4e4f265ea │ 2026-03-01 │ 2026-03-31 │ 869640 │
│ client_08a6a72ff48e62c0 │ 2026-03-01 │ 2026-03-31 │ 851275 │
│ client_62f4a7e64f5e0096 │ 2026-03-01 │ 2026-0

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps
This data can never tell me [X]% of rows are GSC-only (ga4_data_available = FALSE) — those
rows can't support any engagement-based feature. History depth is unbalanced across clients
(per the lane guide, only 9 of 70 clients have 12+ months) — a client with a few weeks of
history can't support a 90-day feature window, so any model trained across all clients will
implicitly know less about short-history clients. The `_sample` table is June 2026 only and
is reserved as a sealed test month — nothing in this contract was developed against it..*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# How many clients actually have enough history for a 90-day feature window
# within this month's partition alone (a lower bound -- full check needs dim_clients).
con.sql(f"""
    SELECT
        COUNT(*) FILTER (WHERE days_of_history >= 90) AS clients_with_90d,
        COUNT(*) AS total_clients
    FROM (
        SELECT client_hash_id, DATE_DIFF('day', MIN(report_date), MAX(report_date)) AS days_of_history
        FROM {tbl}
        GROUP BY 1
    )
""").show()

┌──────────────────┬───────────────┐
│ clients_with_90d │ total_clients │
│      int64       │     int64     │
├──────────────────┼───────────────┤
│                0 │            55 │
└──────────────────┴───────────────┘



## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.